This is a notebook to visualise the spectroscopy data from the USGS data set

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import re
import glob


In [5]:
# Load hyperspectral wavelengths ()
data_dir = Path("ASCIIdata/ASCIIdata_splib07b_rsWorldView3/")
hp_wavelength_pattern = f"*Wavelengths_*S*R*F*.txt"
hp_wavelength_files = list(data_dir.glob(hp_wavelength_pattern))
hp_wavelength_files

[WindowsPath('ASCIIdata/ASCIIdata_splib07b_rsWorldView3/S07WV3_Wavelengths_WorldView3_SpecRespFunctions.txt')]

In [ ]:
class USGSSatelliteSpectra:
    def __init__(self, base_dir, satellite='ASTER'):
        """
        Initialize the USGS Spectral Library satellite data loader
        
        Parameters:
        -----------
        base_dir : str
            Base directory containing USGS Spectral Library files
        satellite : str
            Satellite sensor name (e.g., 'ASTER', 'LSAT8', 'SNTL2', 'WV3')
        """
        self.base_dir = Path(base_dir)
        self.satellite = satellite
        self.prefix = f"S07{satellite}_"
        
        # Find the appropriate data directory
        self.data_dir = list(self.base_dir.glob(f"**/ASCIIdata_splib07b_rs{satellite}"))[0]
        if not self.data_dir.exists():
            raise FileNotFoundError(f"Could not find data directory for {satellite}")
        
        print(f"Found data directory: {self.data_dir}")
        
        # Initialise collections
        self.spectra = {}
        self.wavelengths = None           # Normal wavelengths (for plotting with spectra)
        self.wavelengths_hp = None        # Hyperspectral wavelengths (for band data)
        self.bandpass_nm = None           # Bandpass in nanometers
        self.bandpass_micron = None       # Bandpass in microns
        self.bands = {}                   # Dictionary to store individual band response functions
        
        # Load wavelength data
        self._load_wavelength_data()
        #self._load_band_data()
    
    def _load_wavelength_data(self):
        """Load wavelength and bandpass data for the satellite sensor"""
        
        # Load normal wavelengths (with "bands" in filename)
        wavelength_pattern = f"{self.prefix}Wavelengths*bands*.txt"
        wavelength_files = list(self.data_dir.glob(wavelength_pattern))
        
        if wavelength_files:
            wavelength_file = wavelength_files[0]
            print(f"Loading normal wavelength data from: {wavelength_file}")
            self.wavelengths = np.loadtxt(wavelength_file, skiprows=1)
            print(f"Loaded normal wavelength data: {len(self.wavelengths)} bands")
        
        # Load hyperspectral wavelengths ()
        hp_wavelength_pattern = f"{self.prefix}Wavelengths_*S*R*F*.txt"
        hp_wavelength_files = list(self.data_dir.glob(hp_wavelength_pattern))
        
        if hp_wavelength_files:
            hp_wavelength_file = hp_wavelength_files[0]
            print(f"Loading hyperspectral wavelength data from: {hp_wavelength_file}")
            self.wavelengths_hp = np.loadtxt(hp_wavelength_file, skiprows=1)
            print(f"Loaded hyperspectral wavelength data: {len(self.wavelengths_hp)} channels")
        
        # Load bandpass data in nanometers
        bandpass_nm_pattern = f"{self.prefix}Bandpass*nm.txt"
        bandpass_nm_files = list(self.data_dir.glob(bandpass_nm_pattern))
        
        if bandpass_nm_files:
            bandpass_nm_file = bandpass_nm_files[0]
            print(f"Loading bandpass data (nm) from: {bandpass_nm_file}")
            self.bandpass_nm = np.loadtxt(bandpass_nm_file, skiprows=1)
            print(f"Loaded bandpass data (nm): {len(self.bandpass_nm)} values")
        
        # Load bandpass data in microns
        bandpass_micron_pattern = f"{self.prefix}Bandpass*um.txt"
        bandpass_micron_files = list(self.data_dir.glob(bandpass_micron_pattern))
        
        if bandpass_micron_files:
            bandpass_micron_file = bandpass_micron_files[0]
            print(f"Loading bandpass data (microns) from: {bandpass_micron_file}")
            self.bandpass_micron = np.loadtxt(bandpass_micron_file, skiprows=1)
            print(f"Loaded bandpass data (microns): {len(self.bandpass_micron)} values")
    
    